# Bài 3: Phát hiện Tấn công ICS bằng Mô hình LSTM Có Giám sát

## Mục tiêu
- Xây dựng LSTM có giám sát cho phân loại chuỗi thời gian
- Tạo dữ liệu sliding window từ cảm biến BATADAL
- Huấn luyện và tinh chỉnh ngưỡng để tối ưu F1

## Hướng dẫn
Điền code vào các ô `???`. Tham khảo `b3_lstm_demo.ipynb` nếu cần.

## Phần 1: Import thư viện

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (f1_score, precision_score, recall_score,
                              confusion_matrix, roc_auc_score, roc_curve,
                              ConfusionMatrixDisplay)
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

np.random.seed(42)
tf.random.set_seed(42)
plt.rcParams['figure.figsize'] = (13, 5)
print(f'TensorFlow version: {tf.__version__}')
print('Thư viện đã được import thành công!')

## Phần 2: Tải và Chuẩn bị Dữ liệu

In [ ]:
train_df = pd.read_csv('data/BATADAL/train_dataset.csv', index_col=0)
test1_df = pd.read_csv('data/BATADAL/test_dataset_1.csv', index_col=0)
test2_df = pd.read_csv('data/BATADAL/test_dataset_2.csv', index_col=0)

feature_cols = [c for c in train_df.columns if c not in ['DATETIME', 'ATT_FLAG']]
n_features   = len(feature_cols)

# Chuẩn hóa: fit trên test1 (có nhãn), transform cả test1 và test2
scaler        = StandardScaler()
X_test1_sc    = scaler.fit_transform(test1_df[feature_cols].values)
X_test2_sc    = scaler.transform(test2_df[feature_cols].values)
y_test1       = test1_df['ATT_FLAG'].values
y_test2       = test2_df['ATT_FLAG'].values

print(f'Test 1: {test1_df.shape[0]} mẫu (Normal={(y_test1==0).sum()}, Attack={(y_test1==1).sum()})')
print(f'Test 2: {test2_df.shape[0]} mẫu (Normal={(y_test2==0).sum()}, Attack={(y_test2==1).sum()})')
print(f'Số đặc trưng: {n_features}')

## Phần 3: Tạo Chuỗi bằng Cửa sổ Trượt (Sliding Window)

Mỗi chuỗi gồm `WINDOW_SIZE=30` bước thời gian liên tiếp.
Nhãn của chuỗi = **1** nếu BẤT KỲ bước thời gian nào trong cửa sổ là tấn công.

In [ ]:
WINDOW_SIZE = 30
STEP        = 5   # Bước trượt (stride) khi tạo chuỗi

def create_sequences(X, y, window_size=30, step=1):
    """Tạo chuỗi thời gian với nhãn majority vote."""
    sequences, labels = [], []
    # TODO: Điền bước lặp đúng — dùng biến `step` làm stride
    for i in range(0, len(X) - window_size + 1, ???):
        # TODO: Lấy đoạn X từ i đến i+window_size
        seq   = X[i : i + ???]
        # TODO: Nhãn = 1 nếu có bất kỳ bước nào là tấn công
        label = 1 if y[i : i + window_size].sum() ??? 0 else 0
        sequences.append(seq)
        labels.append(label)
    return np.array(sequences, dtype=np.float32), np.array(labels, dtype=np.int32)

# Tạo chuỗi từ test1 (dùng làm tập train supervised)
X_seq, y_seq = create_sequences(X_test1_sc, y_test1, WINDOW_SIZE, STEP)

# TODO: Chia train/val với tỷ lệ test 30% và stratify
X_tr, X_val, y_tr, y_val = train_test_split(
    X_seq, y_seq, test_size=???, random_state=42, stratify=y_seq
)

# Tạo chuỗi từ test2 (held-out, step=1)
X_test2_seq, y_test2_seq = create_sequences(X_test2_sc, y_test2, WINDOW_SIZE, step=1)

print(f'Chuỗi train : {X_tr.shape}  nhãn attack={(y_tr==1).sum()} ({100*(y_tr==1).mean():.1f}%)')
print(f'Chuỗi val   : {X_val.shape}  nhãn attack={(y_val==1).sum()} ({100*(y_val==1).mean():.1f}%)')
print(f'Chuỗi test2 : {X_test2_seq.shape} nhãn attack={(y_test2_seq==1).sum()} ({100*(y_test2_seq==1).mean():.1f}%)')

## Phần 4: Xây dựng Kiến trúc LSTM

**Kiến trúc LSTM Classifier:**
- Input: (batch, WINDOW_SIZE=30, n_features=43)
- LSTM(64, return_sequences=True) → mỗi bước thời gian đều có output
- Dropout(0.3)
- LSTM(32) → chỉ output bước cuối
- Dropout(0.3)
- Dense(16, relu) → bottleneck
- Dense(1, sigmoid) → xác suất tấn công

In [ ]:
def build_lstm_classifier(n_timesteps, n_features, lstm_units=64):
    model = Sequential([
        # TODO: LSTM đầu tiên — cần trả về chuỗi đầy đủ cho LSTM tiếp theo
        LSTM(lstm_units, return_sequences=???,
             input_shape=(n_timesteps, n_features)),
        Dropout(0.3),
        # TODO: LSTM thứ hai — units = lstm_units chia đôi
        LSTM(lstm_units // ???),
        Dropout(0.3),
        Dense(16, activation='relu'),
        Dense(1, activation='sigmoid')
    ], name='LSTM_Classifier')

    model.compile(
        optimizer=Adam(learning_rate=1e-3),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

model = build_lstm_classifier(WINDOW_SIZE, n_features)
model.summary()

## Phần 5: Huấn luyện Mô hình

In [ ]:
# Class weight để xử lý mất cân bằng nhãn
pos_weight   = (y_tr == 0).sum() / max((y_tr == 1).sum(), 1)
# TODO: Gán trọng số cho lớp tấn công (1) = float(pos_weight)
class_weight = {0: 1.0, 1: ???}
print(f'Class weight: normal=1.0, attack={pos_weight:.2f}')

callbacks = [
    EarlyStopping(patience=10, restore_best_weights=True,
                  monitor='val_loss', verbose=1),
    ReduceLROnPlateau(factor=0.5, patience=5,
                      min_lr=1e-6, verbose=0)
]

# TODO: Truyền tập train vào model.fit
history = model.fit(
    ???, y_tr,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=64,
    class_weight=class_weight,
    callbacks=callbacks,
    verbose=1
)
print(f'\nHuấn luyện kết thúc ở epoch {len(history.history["loss"])}')

In [ ]:
# Learning curves
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(history.history['loss'],     label='Train Loss', color='steelblue')
axes[0].plot(history.history['val_loss'], label='Val Loss',   color='crimson')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss Curve'); axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history['accuracy'],     label='Train Acc', color='steelblue')
axes[1].plot(history.history['val_accuracy'], label='Val Acc',   color='crimson')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
axes[1].set_title('Training Accuracy Curve'); axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

## Phần 6: Tinh chỉnh Ngưỡng và Đánh giá

In [ ]:
# Tìm ngưỡng tối ưu trên tập val
y_prob_val  = model.predict(X_val, verbose=0).ravel()
thresholds  = np.arange(0.1, 0.9, 0.05)
# TODO: Tính F1 cho mỗi ngưỡng t — dùng (y_prob_val > ???) để tạo nhãn dự đoán
f1s         = [f1_score(y_val, (y_prob_val > ???).astype(int),
                        average='macro', zero_division=0)
               for t in thresholds]
# TODO: Chọn ngưỡng có F1 cao nhất — dùng np.argmax(???)
best_thr    = float(thresholds[np.argmax(???)])

plt.figure(figsize=(9, 4))
plt.plot(thresholds, f1s, 'o-', color='steelblue', linewidth=2)
plt.axvline(best_thr, color='crimson', linestyle='--', linewidth=2,
            label=f'Best threshold = {best_thr:.2f}')
plt.xlabel('Ngưỡng quyết định'); plt.ylabel('F1 (macro)')
plt.title('F1-score theo Ngưỡng quyết định (Validation)')
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

# Đánh giá trên val
y_pred_val = (y_prob_val > best_thr).astype(int)
f1  = f1_score(y_val, y_pred_val, average='macro', zero_division=0)
pre = precision_score(y_val, y_pred_val, pos_label=1, zero_division=0)
rec = recall_score(y_val, y_pred_val, pos_label=1, zero_division=0)
auc = roc_auc_score(y_val, y_prob_val)
print(f'\n=== LSTM Classifier (Val, thr={best_thr:.2f}) ===')
print(f'  F1 (macro): {f1:.4f} | Precision: {pre:.4f} | Recall: {rec:.4f}')
print(f'  AUC-ROC  : {auc:.4f}')
cm = confusion_matrix(y_val, y_pred_val)
print(f'  TN={cm[0,0]}, FP={cm[0,1]}, FN={cm[1,0]}, TP={cm[1,1]}')

## Phần 7: Đánh giá trên Test 2 (Held-out)

In [ ]:
# TODO: Dự đoán xác suất cho tập test2 — dùng model.predict(???)
y_prob_test2 = model.predict(???, verbose=0).ravel()
# TODO: Áp dụng ngưỡng tốt nhất — (y_prob_test2 > ???).astype(int)
y_pred_test2 = (y_prob_test2 > ???).astype(int)

f1  = f1_score(y_test2_seq, y_pred_test2, average='macro', zero_division=0)
pre = precision_score(y_test2_seq, y_pred_test2, pos_label=1, zero_division=0)
rec = recall_score(y_test2_seq, y_pred_test2, pos_label=1, zero_division=0)
auc = roc_auc_score(y_test2_seq, y_prob_test2)
print(f'=== LSTM Classifier (Test2, thr={best_thr:.2f}) ===')
print(f'  F1 (macro): {f1:.4f} | Precision: {pre:.4f} | Recall: {rec:.4f}')
print(f'  AUC-ROC  : {auc:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Confusion matrix
cm = confusion_matrix(y_test2_seq, y_pred_test2)
ConfusionMatrixDisplay(cm, display_labels=['Bình thường', 'Tấn công']).plot(
    ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title(f'Confusion Matrix — LSTM (Test 2, thr={best_thr:.2f})')

# ROC Curve
fpr, tpr, _ = roc_curve(y_test2_seq, y_prob_test2)
axes[1].plot(fpr, tpr, color='steelblue', linewidth=2.5,
             label=f'LSTM (AUC={auc:.3f})')
axes[1].plot([0,1],[0,1],'k--', linewidth=1)
axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR')
axes[1].set_title('ROC Curve — LSTM (Test 2)'); axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

## Câu hỏi thảo luận

1. Tại sao dùng `return_sequences=True` ở LSTM đầu tiên nhưng không dùng ở LSTM thứ hai?
2. `class_weight` ảnh hưởng đến quá trình huấn luyện như thế nào khi dữ liệu mất cân bằng?
3. Ngưỡng quyết định tối ưu có phải luôn là 0.5 không? Tại sao?
4. So sánh LSTM có giám sát với LSTM Autoencoder: khi nào dùng cái nào?